[![Open In Colab](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/badge/open-in-colab.svg)](https://colab.research.google.com/github/crunchdao/quickstarters/blob/master/competitions/structural-break/quickstarters/baseline/baseline.ipynb)
[![Open In Kaggle](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/badge/open-in-kaggle.svg)](https://www.kaggle.com/code/crunchdao/structural-break-baseline)

![Banner](https://raw.githubusercontent.com/crunchdao/quickstarters/refs/heads/master/competitions/structural-break/assets/banner.webp)

# ADIA Lab Structural Break Challenge

## Challenge Overview

Welcome to the ADIA Lab Structural Break Challenge! In this challenge, you will analyze univariate time series data to determine whether a structural break has occurred at a specified boundary point.

### What is a Structural Break?

A structural break occurs when the process governing the data generation changes at a certain point in time. These changes can be subtle or dramatic, and detecting them accurately is crucial across various domains such as climatology, industrial monitoring, finance, and healthcare.

![Structural Break Example](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/competitions/structural-break/quickstarters/baseline/images/example.png)

### Your Task

For each time series in the test set, you need to predict a score between `0` and `1`:
- Values closer to `0` indicate no structural break at the specified boundary point;
- Values closer to `1` indicate a structural break did occur.

### Evaluation Metric

The evaluation metric is [ROC AUC (Area Under the Receiver Operating Characteristic Curve)](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html), which measures the performance of detection algorithms regardless of their specific calibration.

- ROC AUC around `0.5`: No better than random chance;
- ROC AUC approaching `1.0`: Perfect detection.

# Setup

The first steps to get started are:
1. Get the setup command
2. Execute it in the cell below

### >> https://hub.crunchdao.com/competitions/structural-break/submit/notebook

![Reveal token](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/reveal-token.gif)

In [18]:
# Install the Crunch CLI
%pip install --upgrade crunch-cli

# Setup your local environment
#!crunch setup --notebook structural-break hello --token aaaabbbbccccddddeeeeffff
!crunch setup-notebook structural-break Y5AcL5oIf905ChiIndjKLTjV

5268.69s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Note: you may need to restart the kernel to use updated packages.


5274.83s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


crunch-cli, version 7.4.0
you appear to have never submitted code before
data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/X_train.parquet (204327238 bytes)
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/X_test.reduced.parquet (2380918 bytes)
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/y_train.parquet (61003 bytes)
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/y_test.reduced.parquet (2655 bytes)
                                                                                
---
Success! Your environment has been correctly setup.
Next recommended actions:
1. Load the Crunch Toolings: `crunch = crunch.load_notebook()`
2. Execute the cells with your code
3. Run a test: `crun

# Your model

## Setup

In [14]:
import os
import typing

# Import your dependencies
import joblib
import pandas as pd
import scipy
import sklearn.metrics

In [19]:
import crunch

# Load the Crunch Toolings
crunch = crunch.load_notebook()

loaded inline runner with module: <module '__main__'>

cli version: 7.4.0
available ram: 30.51 gb
available cpu: 16 core
----


## Understanding the Data

The dataset consists of univariate time series, each containing ~2,000-5,000 values with a designated boundary point. For each time series, you need to determine whether a structural break occurred at this boundary point.

The data was downloaded when you setup your local environment and is now available in the `data/` directory.

In [20]:
# Load the data simply
X_train, y_train, X_test = crunch.load_data()

data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/X_train.parquet (204327238 bytes)
data/X_train.parquet: already exists, file length match
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/X_test.reduced.parquet (2380918 bytes)
data/X_test.reduced.parquet: already exists, file length match
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/y_train.parquet (61003 bytes)
data/y_train.parquet: already exists, file length match
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/y_test.reduced.parquet (2655 bytes)
data/y_test.reduced.parquet: already exists, file length match


In [5]:
import pandas as pd

data_dir = "/home/endrem/project/structural_break/competitions/competitions/structural-break/data"

# Train
X_train = pd.read_parquet(f"{data_dir}/X_train.parquet")
y_train = pd.read_parquet(f"{data_dir}/y_train.parquet")["structural_breakpoint"]

# Test (group into a list of DataFrames by dataset id, like the platform)
_x_test_df = pd.read_parquet(f"{data_dir}/X_test.reduced.parquet")
X_test = []
for id_val, df in _x_test_df.groupby(_x_test_df.index.get_level_values("id")):
    df.name = id_val
    X_test.append(df)

# Optional: local scoring target for the reduced test
y_test = pd.read_parquet(f"{data_dir}/y_test.reduced.parquet")["structural_breakpoint"]


### Understanding `X_train`

The training data is structured as a pandas DataFrame with a MultiIndex:

**Index Levels:**
- `id`: Identifies the unique time series
- `time`: The timestep within each time series

**Columns:**
- `value`: The actual time series value at each timestep
- `period`: A binary indicator where `0` represents the **period before** the boundary point, and `1` represents the **period after** the boundary point

In [6]:
X_train

value  period
id    time                  
0     0    -0.005564       0
      1     0.003705       0
      2     0.013164       0
      3     0.007151       0
      4    -0.009979       0
...              ...     ...
10000 2134  0.001137       1
      2135  0.003526       1
      2136  0.000687       1
      2137  0.001640       1
      2138  0.001074       1

[23715734 rows x 2 columns]

### Understanding `y_train`

This is a simple `pandas.Series` that tells if a dataset id has a structural breakpoint or not.

**Index:**
- `id`: the ID of the dataset

**Value:**
- `structural_breakpoint`: Boolean indicating whether a structural break occurred (`True`) or not (`False`)

In [7]:
y_train

id
0        False
1        False
2         True
3        False
4        False
         ...  
9996     False
9997     False
9998     False
9999     False
10000     True
Name: structural_breakpoint, Length: 10001, dtype: bool

### Understanding `X_test`

The test data is provided as a **`list` of `pandas.DataFrame`s** with the same format as [`X_train`](#understanding-X_test).

It is structured as a list to encourage processing records one by one, which will be mandatory in the `infer()` function.

In [8]:
print("Number of datasets:", len(X_test))

Number of datasets: 101


In [9]:
X_test[0]

value  period
id    time                  
10001 0     0.010753       0
      1    -0.031915       0
      2    -0.010989       0
      3    -0.011111       0
      4     0.011236       0
...              ...     ...
      2774 -0.013937       1
      2775 -0.015649       1
      2776 -0.009744       1
      2777  0.025375       1
      2778 -0.001515       1

[2779 rows x 2 columns]

## Strategy Implementation

There are multiple approaches you can take to detect structural breaks:

1. **Statistical Tests**: Compare distributions before and after the boundary point;
2. **Feature Engineering**: Extract features from both segments for comparison;
3. **Time Series Modeling**: Detect deviations from expected patterns;
4. **Machine Learning**: Train models to recognize break patterns from labeled examples.

The baseline implementation below uses a simple statistical approach: a t-test to compare the distributions before and after the boundary point.

### The `train()` Function

In this function, you build and train your model for making inferences on the test data. Your model must be stored in the `model_directory_path`.

The baseline implementation below doesn't require a pre-trained model, as it uses a statistical test that will be computed at inference time.

In [32]:
import os
import typing
import numpy as np
import pandas as pd
import scipy.stats as st
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

FEATURES_TEST = [
    "mean_diff", "abs_mean_diff", "median_diff",
    "std_ratio", "mad_ratio",
    "skew_diff", "kurt_diff",
    "acf1_diff",
    "p_t_welch", "p_ks", "p_mw", "p_levene",
]

def _mad(x):
    m = np.median(x) if len(x) else 0.0
    return np.median(np.abs(x - m)) if len(x) else 0.0

def _acf1(x):
    if len(x) < 2:
        return 0.0
    s = pd.Series(x)
    v = s.autocorr(lag=1)
    return 0.0 if pd.isna(v) else float(v)

def _logp(p):
    p = np.clip(p, 1e-300, 1.0)
    return -np.log10(p)

def _fe_one(u: pd.DataFrame) -> np.ndarray:
    a = u["value"][u["period"] == 0].to_numpy()
    b = u["value"][u["period"] == 1].to_numpy()
    eps = 1e-12

    # robust stats
    mean_a, mean_b = (a.mean() if a.size else 0.0), (b.mean() if b.size else 0.0)
    med_a,  med_b  = (np.median(a) if a.size else 0.0), (np.median(b) if b.size else 0.0)
    std_a,  std_b  = (a.std(ddof=1) if a.size>1 else 0.0), (b.std(ddof=1) if b.size>1 else 0.0)
    mad_a,  mad_b  = _mad(a), _mad(b)
    skew_a, skew_b = (st.skew(a) if a.size>2 else 0.0), (st.skew(b) if b.size>2 else 0.0)
    kur_a,  kur_b  = (st.kurtosis(a) if a.size>3 else 0.0), (st.kurtosis(b) if b.size>3 else 0.0)
    acf_a,  acf_b  = _acf1(a), _acf1(b)

    # tests
    p_t = st.ttest_ind(a, b, equal_var=False).pvalue if a.size and b.size else 1.0
    p_ks = st.ks_2samp(a, b, alternative="two-sided", mode="asymp").pvalue if a.size and b.size else 1.0
    p_mw = st.mannwhitneyu(a, b, alternative="two-sided").pvalue if a.size and b.size else 1.0
    p_lv = st.levene(a, b, center="median").pvalue if a.size and b.size else 1.0

    feats = [
        (mean_b - mean_a),
        abs(mean_b - mean_a),
        (med_b - med_a),
        (std_b + eps) / (std_a + eps),
        (mad_b + eps) / (mad_a + eps),
        (skew_b - skew_a),
        (kur_b - kur_a),
        (acf_b - acf_a),
        _logp(p_t), _logp(p_ks), _logp(p_mw), _logp(p_lv),
    ]
    # sanitize
    feats = [0.0 if (isinstance(v, float) and (np.isnan(v) or np.isinf(v))) else v for v in feats]
    return np.asarray(feats, dtype=float)

def _fe_table(X_train: pd.DataFrame, FEATURES_TEST) -> pd.DataFrame:
    rows, ids = [], []
    for gid, gdf in X_train.groupby(level="id"):
        rows.append(_fe_one(gdf))
        ids.append(gid)
    return pd.DataFrame(rows, index=pd.Index(ids, name="id"), columns=FEATURES_TEST)


In [33]:
def train(X_train: pd.DataFrame, y_train: pd.Series, model_directory_path: str):
    FEATURES_TEST = [
        "mean_diff", "abs_mean_diff", "median_diff",
        "std_ratio", "mad_ratio",
        "skew_diff", "kurt_diff",
        "acf1_diff",
        "p_t_welch", "p_ks", "p_mw", "p_levene",
    ]
    
    F = _fe_table(X_train, FEATURES_TEST).reindex(y_train.index)
    y = y_train.astype(int).to_numpy()
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=500, class_weight="balanced", n_jobs=None)),
    ])
    pipe.fit(F.values, y)
    joblib.dump({"model": pipe, "features": FEATURES_TEST}, os.path.join(model_directory_path, "model.joblib"))


In [26]:
def train2(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    model_directory_path: str,
):
    # For our baseline t-test approach, we don't need to train a model
    # This is essentially an unsupervised approach calculated at inference time
    model = None

    # You could enhance this by training an actual model, for example:
    # 1. Extract features from before/after segments of each time series
    # 2. Train a classifier using these features and y_train labels
    # 3. Save the trained model

    joblib.dump(model, os.path.join(model_directory_path, 'model.joblib'))

### The `infer()` Function

In the inference function, your trained model (if any) is loaded and used to make predictions on test data.

**Important workflow:**
1. Load your model;
2. Use the `yield` statement to signal readiness to the runner;
3. Process each dataset one by one within the for loop;
4. For each dataset, use `yield prediction` to return your prediction.

**Note:** The datasets can only be iterated once!

In [34]:
def infer(X_test: typing.Iterable[pd.DataFrame], model_directory_path: str):
    bundle = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    model: Pipeline = bundle["model"]
    yield  # ready
    for dataset in X_test:
        f = _fe_one(dataset).reshape(1, -1)
        p = float(model.predict_proba(f)[0, 1])  # probability of break
        yield p

In [28]:
def infer2(
    X_test: typing.Iterable[pd.DataFrame],
    model_directory_path: str,
):
    model = joblib.load(os.path.join(model_directory_path, 'model.joblib'))

    yield  # Mark as ready

    # X_test can only be iterated once.
    # Before getting the next dataset, you must predict the current one.
    for dataset in X_test:
        # Baseline approach: Compute t-test between values before and after boundary point
        # The negative p-value is used as our score - smaller p-values (larger negative numbers)
        # indicate more evidence against the null hypothesis that distributions are the same,
        # suggesting a structural break
        def t_test(u: pd.DataFrame):
            return -scipy.stats.ttest_ind(
                u["value"][u["period"] == 0],  # Values before boundary point
                u["value"][u["period"] == 1],  # Values after boundary point
            ).pvalue

        prediction = t_test(dataset)
        yield prediction  # Send the prediction for the current dataset

        # Note: This baseline approach uses a t-test to compare the distributions
        # before and after the boundary point. A smaller p-value (larger negative number)
        # suggests stronger evidence that the distributions are different,
        # indicating a potential structural break.

## Local testing

To make sure your `train()` and `infer()` function are working properly, you can call the `crunch.test()` function that will reproduce the cloud environment locally. <br />
Even if it is not perfect, it should give you a quick idea if your model is working properly.

In [35]:
crunch.test(
    # Uncomment to disable the train
    # force_first_train=False,

    # Uncomment to disable the determinism check
    # no_determinism_check=True,
)

15:14:28 no forbidden library found
15:14:28 
15:14:28 started
15:14:28 running local test
15:14:28 internet access isn't restricted, no check will be done
15:14:28 
15:14:29 starting unstructured loop...
15:14:29 executing - command=train


data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/X_train.parquet (204327238 bytes)
data/X_train.parquet: already exists, file length match
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/X_test.reduced.parquet (2380918 bytes)
data/X_test.reduced.parquet: already exists, file length match
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/y_train.parquet (61003 bytes)
data/y_train.parquet: already exists, file length match
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/y_test.reduced.parquet (2655 bytes)
data/y_test.reduced.parquet: already exists, file length match


15:15:21 executing - command=infer
15:15:22 checking determinism by executing the inference again with 30% of the data (tolerance: 1e-08)
15:15:22 executing - command=infer
15:15:22 determinism check: passed
15:15:22 save prediction - path=data/prediction.parquet
15:15:22 ended
15:15:22 duration - time=00:00:53
15:15:22 memory - before="6.5 GB" after="7.7 GB" consumed="1.2 GB"


## Results

Once the local tester is done, you can preview the result stored in `data/prediction.parquet`.

In [36]:
prediction = pd.read_parquet("data/prediction.parquet")
prediction

,prediction
id,
10001,0.451047
10002,0.451763
10003,0.500941
10004,0.418656
10005,0.442000
...,...
10097,0.444516
10098,0.436356
10099,0.572405


### Local scoring

You can call the function that the system uses to estimate your score locally.

In [37]:
# Load the targets
target = pd.read_parquet("data/y_test.reduced.parquet")["structural_breakpoint"]

# Call the scoring function
sklearn.metrics.roc_auc_score(
    target,
    prediction,
)

0.5150234741784037

# Submit your Notebook

To submit your work, you must:
1. Download your Notebook from Colab
2. Upload it to the platform
3. Create a run to validate it

### >> https://hub.crunchdao.com/competitions/structural-break/submit/notebook

![Download and Submit Notebook](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/download-and-submit-notebook.gif)